# Merge Panel: VIIRS + OSM + Eurostat

This notebook constructs the **analysis-ready panel dataset** by merging three sources:

| Source | Key | Rows | Type | Content |
|--------|-----|------|------|---------|
| VIIRS nighttime lights | `(city, year)` | 360 | Panel | Dependent variable: nighttime brightness |
| Eurostat | `(city, year)` | 360 | Panel | Population, GDP per capita, density |
| OSM features | `(city)` | 30 | Cross-sectional | Urban morphology: roads, buildings, land use, green areas |

**Merge strategy:**
```
VIIRS (360 rows)
  LEFT JOIN Eurostat ON (city, year)
  LEFT JOIN OSM ON (city)  -- broadcasts across all years
  = Final panel (360 rows, ~30 columns)
```

After merging, the notebook:
- Validates the join (no row gain/loss, key integrity)
- Engineers derived features (log transforms, growth rates)
- Profiles data quality (missingness, distributions, correlations)
- Saves to `../data/processed/analysis_panel.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)

## 1. Load sources

In [ ]:
viirs = pd.read_csv("../data/processed/viirs_city_panel_clean.csv")
osm = pd.read_csv("../data/processed/osm_city_features.csv")
euro = pd.read_csv("../data/processed/eurostat_city_panel.csv")

print(f"VIIRS:    {viirs.shape[0]:>4} rows x {viirs.shape[1]:>2} cols  |  {viirs['city'].nunique()} cities  |  {viirs['year'].min()}-{viirs['year'].max()}")
print(f"OSM:      {osm.shape[0]:>4} rows x {osm.shape[1]:>2} cols  |  {osm['city'].nunique()} cities  |  cross-sectional")
print(f"Eurostat: {euro.shape[0]:>4} rows x {euro.shape[1]:>2} cols  |  {euro['city'].nunique()} cities  |  {euro['year'].min()}-{euro['year'].max()}")

## 2. Pre-merge validation

In [ ]:
v_cities = set(viirs["city"])
o_cities = set(osm["city"])
e_cities = set(euro["city"])

assert v_cities == o_cities == e_cities, (
    f"City mismatch! "
    f"VIIRS-OSM diff: {v_cities.symmetric_difference(o_cities)}, "
    f"VIIRS-Euro diff: {v_cities.symmetric_difference(e_cities)}"
)
print(f"OK: All 3 sources share the same {len(v_cities)} cities.")

assert viirs.duplicated(subset=["city", "year"]).sum() == 0, "VIIRS has duplicate (city, year)"
assert euro.duplicated(subset=["city", "year"]).sum() == 0, "Eurostat has duplicate (city, year)"
assert osm.duplicated(subset=["city"]).sum() == 0, "OSM has duplicate city"
print("OK: No duplicate keys in any source.")

v_years = set(viirs["year"])
e_years = set(euro["year"])
assert v_years == e_years, f"Year mismatch: {v_years.symmetric_difference(e_years)}"
print(f"OK: Both panels cover {min(v_years)}-{max(v_years)} ({len(v_years)} years).")

## 3. Execute merge

In [ ]:
# Step 1: VIIRS as backbone (complete panel, zero NaN)
panel = viirs.copy()
print(f"After VIIRS:     {panel.shape}")

# Step 2: Left-join Eurostat on (city, year)
euro_cols = ["city", "year", "population", "gdp_per_capita_eur", "population_density"]
panel = panel.merge(euro[euro_cols], on=["city", "year"], how="left")
print(f"After Eurostat:  {panel.shape}")

# Step 3: Left-join OSM on (city) -- broadcasts across all years
# Drop buffer_area_km2 (internal to OSM computation, not needed for analysis)
osm_cols = [c for c in osm.columns if c != "buffer_area_km2"]
panel = panel.merge(osm[osm_cols], on="city", how="left")
print(f"After OSM:       {panel.shape}")

assert len(panel) == 360, f"Expected 360 rows, got {len(panel)}"
print(f"\nFinal panel: {panel.shape[0]} rows x {panel.shape[1]} columns")

## 4. Post-merge validation

In [ ]:
print("Panel structure:")
print(f"  Cities: {panel['city'].nunique()}")
print(f"  Years:  {panel['year'].min()}-{panel['year'].max()} ({panel['year'].nunique()} years)")
print(f"  Rows:   {len(panel)} (expected {30 * 12})")
print(f"  Cols:   {panel.shape[1]}")

# Balanced panel check
obs_per_city = panel.groupby("city").size()
if (obs_per_city == 12).all():
    print("\nOK: Balanced panel (12 years for every city).")
else:
    print("\nWARNING: Unbalanced panel.")
    print(obs_per_city[obs_per_city != 12])

# Missingness summary
print("\nMissingness:")
missing = panel.isna().sum()
missing_pct = (missing / len(panel) * 100).round(1)
miss_df = pd.DataFrame({"n_missing": missing, "pct": missing_pct})
print(miss_df[miss_df["n_missing"] > 0].to_string() if miss_df["n_missing"].sum() > 0 else "  None")

In [ ]:
# Missingness heatmap by city
check_cols = ["population", "gdp_per_capita_eur", "population_density"]
available = [c for c in check_cols if c in panel.columns]

if available:
    miss_by_city = panel.groupby("city")[available].apply(lambda x: x.isna().sum())
    miss_by_city = miss_by_city.loc[miss_by_city.sum(axis=1) > 0]

    if len(miss_by_city) > 0:
        fig, ax = plt.subplots(figsize=(8, max(4, len(miss_by_city) * 0.4)))
        sns.heatmap(miss_by_city, annot=True, fmt="d", cmap="YlOrRd", ax=ax)
        ax.set_title("Missing Eurostat observations per city (out of 12 years)")
        plt.tight_layout()
        plt.show()
    else:
        print("No Eurostat missingness by city.")

## 5. Feature engineering

Derived variables that multiple downstream notebooks will use:
- **Log transforms** of right-skewed variables (brightness, population, GDP, building count) for the regression
- **Brightness growth** year-over-year and over the full study period
- **Light per capita** as an intensity-normalized measure

In [ ]:
# Log transforms
panel["log_mean_brightness"] = np.log1p(panel["mean_brightness"])
panel["log_population"] = np.log(panel["population"])
panel["log_gdp_per_capita"] = np.log(panel["gdp_per_capita_eur"])
panel["log_building_count"] = np.log1p(panel["building_count"])

print("Log transforms added.")

In [ ]:
# Year-over-year brightness growth rate per city
panel = panel.sort_values(["city", "year"])
panel["brightness_yoy"] = panel.groupby("city")["mean_brightness"].pct_change()

# Compound annual growth rate (CAGR) of brightness over full period
first_last = panel.groupby("city")["mean_brightness"].agg(["first", "last"])
n_years = panel["year"].max() - panel["year"].min()
first_last["cagr"] = (first_last["last"] / first_last["first"]) ** (1 / n_years) - 1
panel = panel.merge(first_last[["cagr"]].rename(columns={"cagr": "brightness_cagr"}),
                     left_on="city", right_index=True, how="left")

print("Brightness growth rates:")
print(first_last[["cagr"]].sort_values("cagr", ascending=False).round(4).to_string())

In [ ]:
# Light per capita (brightness normalized by population)
panel["brightness_per_capita"] = panel["mean_brightness"] / panel["population"]

print(f"Feature engineering complete. Panel now has {panel.shape[1]} columns.")
print(f"\nColumns:\n{panel.columns.tolist()}")

## 6. Data profile

In [ ]:
# Summary statistics for key variables
key_vars = [
    "mean_brightness", "median_brightness",
    "population", "gdp_per_capita_eur", "population_density",
    "road_length_total_km", "road_density_km_per_km2",
    "building_count", "building_area_km2",
    "landuse_residential_km2", "landuse_commercial_km2", "landuse_industrial_km2",
    "green_area_total_km2", "green_fraction",
    "street_lamp_count",
    "brightness_cagr",
]
available_vars = [v for v in key_vars if v in panel.columns]
panel[available_vars].describe().T[["count", "mean", "std", "min", "50%", "max"]].round(2)

In [ ]:
# Distribution of key variables
plot_vars = ["mean_brightness", "log_mean_brightness", "population", "log_population",
             "road_density_km_per_km2", "green_fraction"]
plot_vars = [v for v in plot_vars if v in panel.columns]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, var in zip(axes.flat, plot_vars):
    panel[var].dropna().hist(bins=30, ax=ax, edgecolor="black", alpha=0.7)
    ax.set_title(var, fontsize=10)
    ax.set_ylabel("count")
for ax in axes.flat[len(plot_vars):]:
    ax.set_visible(False)
plt.suptitle("Distributions of key variables", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix of cross-sectional predictors (using city means for panel vars)
city_means = panel.groupby("city").mean(numeric_only=True)

corr_vars = [
    "mean_brightness",
    "population", "gdp_per_capita_eur", "population_density",
    "road_density_km_per_km2", "building_count", "building_area_km2",
    "landuse_residential_km2", "landuse_commercial_km2", "landuse_industrial_km2",
    "green_fraction", "street_lamp_count",
]
corr_vars = [v for v in corr_vars if v in city_means.columns]

fig, ax = plt.subplots(figsize=(12, 10))
corr = city_means[corr_vars].corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, ax=ax, square=True)
ax.set_title("Correlation matrix (city-level means)", fontsize=13)
plt.tight_layout()
plt.show()

## 7. Spot-check: brightness vs key predictors

In [ ]:
# Scatter plots: mean brightness vs top predictors (city-level averages)
scatter_vars = ["population", "gdp_per_capita_eur", "road_density_km_per_km2",
                "building_count", "green_fraction", "street_lamp_count"]
scatter_vars = [v for v in scatter_vars if v in city_means.columns]

n = len(scatter_vars)
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, var in zip(axes.flat, scatter_vars):
    data = city_means[["mean_brightness", var]].dropna()
    ax.scatter(data[var], data["mean_brightness"], alpha=0.7, edgecolor="black")
    for city_name in ["Paris", "Madrid", "Oslo", "Tallinn", "Brussels"]:
        if city_name in data.index:
            ax.annotate(city_name, (data.loc[city_name, var], data.loc[city_name, "mean_brightness"]),
                        fontsize=7, alpha=0.7)
    ax.set_xlabel(var, fontsize=9)
    ax.set_ylabel("mean_brightness")
for ax in axes.flat[n:]:
    ax.set_visible(False)
plt.suptitle("Mean brightness vs predictors (city-level averages)", fontsize=13)
plt.tight_layout()
plt.show()

## 8. Save analysis-ready panel

In [ ]:
output_path = "../data/processed/analysis_panel.csv"
panel.to_csv(output_path, index=False)
print(f"Saved {panel.shape[0]} rows x {panel.shape[1]} columns to {output_path}")

## 9. Data dictionary

### Identifiers
| Column | Type | Description |
|--------|------|-------------|
| `city` | str | City name (primary key with `year`) |
| `country` | str | Country name |
| `country_code` | str | ISO 2-letter country code |
| `year` | int | Year (2013-2024) |
| `lat` | float | City center latitude |
| `lon` | float | City center longitude |

### Dependent variables (VIIRS)
| Column | Type | Description |
|--------|------|-------------|
| `mean_brightness` | float | Mean radiance within 25 km buffer (nW/cm2/sr) |
| `median_brightness` | float | Median radiance within 25 km buffer |
| `max_brightness` | float | Maximum radiance within 25 km buffer |
| `pixel_count` | int | Number of VIIRS pixels in the buffer |

### Socioeconomic variables (Eurostat) -- panel
| Column | Type | Description |
|--------|------|-------------|
| `population` | float | Total population (Urban Audit). ~21% missing. |
| `gdp_per_capita_eur` | float | GDP per capita in EUR (NUTS-3). ~8% missing. |
| `population_density` | float | Persons per km2 (NUTS-3, latest year, cross-sectional). ~7% missing. |

### Urban morphology (OSM) -- cross-sectional, repeated across years
| Column | Type | Description |
|--------|------|-------------|
| `road_length_total_km` | float | Total road network length (km) |
| `motorway_length_km` | float | Motorway + trunk road length (km) |
| `road_density_km_per_km2` | float | Road length per buffer area |
| `street_lamp_count` | int | Count of mapped street lamps |
| `building_count` | int | Number of building footprints (25 km count) |
| `building_area_km2` | float | Total building footprint area (10 km core) |
| `landuse_residential_km2` | float | Residential land use area |
| `landuse_commercial_km2` | float | Commercial land use area |
| `landuse_industrial_km2` | float | Industrial land use area |
| `landuse_retail_km2` | float | Retail land use area |
| `park_area_km2` | float | Parks and gardens area |
| `forest_area_km2` | float | Forest and woodland area |
| `water_area_km2` | float | Water body area |
| `nature_reserve_area_km2` | float | Nature reserve area |
| `green_area_total_km2` | float | Sum of park + forest + reserve |
| `built_up_fraction` | float | Built-up area / buffer area |
| `green_fraction` | float | Green area / buffer area |

### Derived features
| Column | Type | Description |
|--------|------|-------------|
| `log_mean_brightness` | float | log(1 + mean_brightness) |
| `log_population` | float | log(population) |
| `log_gdp_per_capita` | float | log(gdp_per_capita_eur) |
| `log_building_count` | float | log(1 + building_count) |
| `brightness_yoy` | float | Year-over-year brightness growth rate |
| `brightness_cagr` | float | Compound annual growth rate (2013-2024) |
| `brightness_per_capita` | float | mean_brightness / population |